# nb14 — Student Distillation Training + Inference (Kaggle / GPU)

Trains the **HDC-RWKV student** with knowledge distillation from the teacher
trained in `nb13`. This is the model that gets deployed to the 6502 / ESP32.

### Pipeline

1. Load teacher + shared BPE from `nb13` outputs
2. Build HDC-RWKV student (bipolar weights via STE)
3. Train with combined loss: 30% NLL + 70% KL(student || teacher) at T=4.0
4. Evaluate hard (deployment-mode) val loss and BPC
5. Generate samples
6. Export deployable binary (`student.bin`)

### Student spec (target ~130 KB for paged ESP32 deployment)

- `vocab=512` (same BPE as teacher)
- `d=1024`, `n_layers=2`
- Deployment: 2 × 512 × 1024 / 8 + 2 × 1024 / 8 = **131,328 bytes** ≈ 128 KB


## Cell 1 — Setup


In [ ]:
import os, sys, subprocess
from pathlib import Path

# Same setup as nb13 — adjust to your Kaggle environment
WOZFORMER_PATH = Path('/kaggle/input/wozformer')

if WOZFORMER_PATH.exists():
    os.chdir(WOZFORMER_PATH)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)

import wozformer as wz
import torch
import math
import matplotlib.pyplot as plt

device = wz.utils.get_device()
print(f'device: {device}')


## Cell 2 — Hyperparameters


In [ ]:
# Student hyperparameters
VOCAB_SIZE = 512    # must match teacher
D          = 1024
N_LAYERS   = 2
BLOCK_SIZE = 64

# Training
BATCH_SIZE = 32     # smaller — recurrence is sequential, memory-heavy
LR         = 3e-3
N_STEPS    = 12000
EVAL_EVERY = 500
SEED       = 1337

# Distillation
TEMP         = 4.0
ALPHA_NLL    = 0.3   # weight on hard-target NLL
ALPHA_DISTIL = 0.7   # weight on soft-target KL

# IO
RUN_DIR    = Path('/kaggle/working/runs')
RUN_DIR.mkdir(parents=True, exist_ok=True)

# Where to find nb13's outputs (upload teacher.pt + bpe.json as Kaggle inputs)
INPUT_DIR  = Path('/kaggle/input/wozformer-teacher')
TEACHER_PT = INPUT_DIR / 'teacher.pt'
BPE_JSON   = INPUT_DIR / 'bpe.json'

# Fallback if you ran nb13 in this same kernel:
if not TEACHER_PT.exists():
    TEACHER_PT = RUN_DIR / 'teacher.pt'
    BPE_JSON   = RUN_DIR / 'bpe.json'

assert TEACHER_PT.exists(), f'Need teacher.pt — got {TEACHER_PT}'
assert BPE_JSON.exists(),   f'Need bpe.json — got {BPE_JSON}'


## Cell 3 — Load tokenizer + corpus + teacher


In [ ]:
wz.utils.set_seed(SEED)

# Tokenizer
tok = wz.tokenizer.BPETokenizer.load(BPE_JSON)
print(f'loaded BPE: vocab={tok.vocab_size}, {len(tok.merges)} merges')

# Corpus
corpus_candidates = [
    WOZFORMER_PATH / 'data' / 'tinyshakespeare.txt',
    Path('/kaggle/input/tinyshakespeare/tinyshakespeare.txt'),
    Path('data/tinyshakespeare.txt'),
]
corpus_path = next((p for p in corpus_candidates if p.exists()), None)
assert corpus_path is not None, 'corpus not found'

text = wz.data.load_corpus(corpus_path)
ids  = torch.tensor(tok.encode(text), dtype=torch.long)
train_data, val_data = wz.data.split_train_val(ids)
print(f'tokens: train {len(train_data):,} / val {len(val_data):,}')

# Teacher
ck = torch.load(TEACHER_PT, map_location=device, weights_only=False)
tcfg = wz.config.TransformerConfig(
    vocab_size=ck['config']['vocab_size'],
    d_model=ck['config']['d_model'],
    num_heads=ck['config']['num_heads'],
    n_layers=ck['config']['n_layers'],
    mlp_mult=ck['config']['mlp_mult'],
)
teacher = wz.models.TinyTransformer(tcfg, block_size=ck['config']['block_size']).to(device)
teacher.load_state_dict(ck['model_state'])
teacher.eval()
print(f'teacher loaded: {wz.utils.count_params(teacher):,} params, best val={ck["best_val"]:.4f}')


## Cell 4 — Build the student


In [ ]:
scfg = wz.config.HDCRWKVConfig(
    vocab_size=VOCAB_SIZE,
    d=D,
    n_layers=N_LAYERS,
    block_size=BLOCK_SIZE,
)
student = wz.models.HDCRWKV(scfg).to(device)

n_train = wz.utils.count_params(student)
n_bytes = student.deployment_bytes()
print(f'student trainable params: {n_train:,}')
print(f'student deployment bytes: {n_bytes:,} ({n_bytes/1024:.1f} KB)')
print(f'compression vs teacher:   {wz.utils.count_params(teacher)*4 / n_bytes:.1f}x (assuming fp32 teacher)')


## Cell 5 — Train with distillation

~20–30 min on a Kaggle T4 (RWKV recurrence is sequential, slower than transformer per step).

### Two losses logged:

- **val(soft)** — STE forward, what training optimizes.
- **val(hard)** — `.sign()` forward, what the 6502 will compute. **This is the number that matters for deployment.**

Healthy run: both descend together with gap < 0.5 nats.


In [ ]:
train_cfg = wz.config.TrainConfig(
    batch_size=BATCH_SIZE,
    block_size=BLOCK_SIZE,
    lr=LR,
    n_steps=N_STEPS,
    eval_every=EVAL_EVERY,
    seed=SEED,
)
dcfg = wz.config.DistillationConfig(
    temperature=TEMP,
    alpha_nll=ALPHA_NLL,
    alpha_distill=ALPHA_DISTIL,
)

history, best = wz.distillation.train_with_distillation(
    student, teacher, train_data, val_data,
    train_cfg, dcfg, device=device, eval_hard=True,
)
print(f'\nbest HARD val: {best["val"]:.4f} at step {best["step"]}')


## Cell 6 — Plot loss curves and report BPC


In [ ]:
steps      = [h[0] for h in history]
vals_soft  = [h[1] for h in history]
vals_hard  = [h[2] for h in history]

plt.figure(figsize=(9, 4))
plt.plot(steps, vals_soft, label='val(soft, STE)', alpha=0.7)
plt.plot(steps, vals_hard, label='val(hard, deploy)', linewidth=2, color='C2')
plt.axhline(ck['best_val'], color='red', linestyle='--', alpha=0.6, label=f'teacher val ({ck["best_val"]:.3f})')
plt.xlabel('step'); plt.ylabel('cross-entropy (nats/token)')
plt.title(f'HDC-RWKV student with distillation (d={D}, V={VOCAB_SIZE})')
plt.legend(); plt.grid(alpha=0.3); plt.show()

# BPC
sample = val_data[:5000].tolist()
avg_cpt = wz.metrics.avg_chars_per_token(tok, sample)
bpc_student = wz.metrics.bits_per_char(best['val'], avg_cpt)
bpc_teacher = wz.metrics.bits_per_char(ck['best_val'], avg_cpt)
print(f'avg chars/token: {avg_cpt:.2f}')
print(f'TEACHER  BPC: {bpc_teacher:.4f}  (fp32, {wz.utils.count_params(teacher):,} params)')
print(f'STUDENT  BPC: {bpc_student:.4f}  (binary, {n_bytes:,} deploy bytes)')
print(f'gap:          {bpc_student - bpc_teacher:+.4f} BPC')


## Cell 7 — Compare generation: teacher vs student


In [ ]:
prompts = ['king', 'romeo', 'my lord,']
seeds   = [1337, 42, 7]

for prompt, seed in zip(prompts, seeds):
    print(f'\n===== prompt: {prompt!r}  seed={seed} =====')
    print('--- teacher (fp32) ---')
    print(wz.generate.generate(
        teacher, tok, prompt=prompt, max_new_tokens=80,
        block_size=ck['config']['block_size'], temperature=0.7, top_k=10,
        seed=seed, device=device, use_hard=False,
    ))
    print('--- student (binary, deployment mode) ---')
    print(wz.generate.generate(
        student, tok, prompt=prompt, max_new_tokens=80,
        block_size=BLOCK_SIZE, temperature=0.7, top_k=10,
        seed=seed, device=device, use_hard=True,
    ))


## Cell 8 — Save student checkpoint + export deployable binary

Two artifacts:

- `student.pt` — Python checkpoint (model state + history + config)
- `student.bin` — bit-packed binary for ESP32 paging / 6502 inference


In [ ]:
import struct
import numpy as np

STUDENT_PT  = RUN_DIR / 'student.pt'
STUDENT_BIN = RUN_DIR / 'student.bin'

torch.save(
    {
        'config': {
            'vocab_size': VOCAB_SIZE, 'd': D, 'n_layers': N_LAYERS,
            'block_size': BLOCK_SIZE,
        },
        'model_state': student.state_dict(),
        'history': history,
        'best_val_hard': best['val'],
        'best_step': best['step'],
        'n_train_params': n_train,
        'deploy_bytes':   n_bytes,
        'teacher_best_val': ck['best_val'],
        'distillation_temp':    TEMP,
        'distillation_alpha_nll':    ALPHA_NLL,
        'distillation_alpha_distil': ALPHA_DISTIL,
    },
    STUDENT_PT,
)
print(f'saved student → {STUDENT_PT}  ({STUDENT_PT.stat().st_size/1024:.1f} KB)')

# Bit-pack into deployment binary
def pack_bits(continuous_tensor):
    binary = (continuous_tensor > 0).to(torch.uint8).cpu().numpy()
    return np.packbits(binary, axis=-1, bitorder='big')

vocab_packed = pack_bits(student.vocab_hv_c.data)
proto_packed = pack_bits(student.prototype_hv_c.data)
decay_packed = [pack_bits(dm.data.unsqueeze(0)).squeeze(0) for dm in student.decay_masks_c]

buf = bytearray()
buf += b'WHRK'
buf += bytes([3, 0])                            # version, reserved
buf += struct.pack('<H', VOCAB_SIZE)
buf += struct.pack('<H', D // 8)
buf += bytes([BLOCK_SIZE, N_LAYERS])
buf += struct.pack('<f', student.log_temp.item())
buf += bytes(2)
buf += vocab_packed.tobytes()
buf += proto_packed.tobytes()
for dp in decay_packed:
    buf += dp.tobytes()

STUDENT_BIN.write_bytes(buf)
print(f'saved binary → {STUDENT_BIN}  ({len(buf):,} bytes = {len(buf)/1024:.1f} KB)')
print(f'\nBoth files ready for download from /kaggle/working/runs/')


## Post-mortem — what you have now

- **Teacher**: dense transformer, ~3M params, fp32. **Lives only in training pipeline.**
- **Student**: HDC-RWKV, ~128 KB binary. **The deployable model.**
- Quality bar set by teacher; student trades quality for binary-only operations and 25× compression.

### Next steps (locally on your machine)

1. Download `student.bin` and `bpe.json` to `wozformer/runs/`.
2. Run `scripts/compare.py --paths runs/teacher.pt runs/student.pt` to see side-by-side BPC.
3. Wire `student.bin` into the ESP32 paging firmware (next milestone).
